# Biolord Train + Condot-Aligned Evaluation (Sciplex3)

默认使用 1000 维输入训练 Biolord，评估时在 DEGs 子空间（通常为 50 维）计算 condot 指标，并保存 test/ood 结果与 UMAP。

In [ ]:
import os
import sys

# ====== GPU 选择 ======
# 必须在 import torch 之前设 CUDA_VISIBLE_DEVICES, 否则 CUDA runtime 已初始化后再改无效.
# 如需换 GPU, 请修改 GPU_ID 并重启内核.
GPU_ID = 1
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

# Clear stale module cache to avoid using old condot_compat code in a long-lived kernel
for _m in list(sys.modules):
    if _m == 'condot_compat' or _m.startswith('condot_compat.'):
        del sys.modules[_m]

import torch
from pathlib import Path

torch.set_float32_matmul_precision('high')

# 确保仓库根目录在 sys.path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from condot_compat import (
    load_adata,
    save_config_snapshot,
    train_biolord_model,
    evaluate_biolord_condot_aligned,
    generate_umap_from_eval_data,
)
from utils.parameters_sciplex3 import module_params, trainer_params

In [ ]:
# ====== 可改参数 ======
# 注: GPU_ID 在上一个 cell 已定义并通过 CUDA_VISIBLE_DEVICES 生效, 这里不再重复设置.
PROCESSED_DATA_PATH = 'results/sciplex3/preprocessed/sciplex3_biolord_condot_aligned.h5ad'
RESULT_ROOT = 'results/sciplex3'
EXPERIMENT_TAG = 'biolord_sciplex3_condot_aligned'

SEED = 42
SPLIT_KEY = 'split_eval'

N_LATENT = 256
EPOCHS = 200
BATCH_SIZE = 512
NUM_WORKERS = 8
CHECKPOINT_FREQ = 10
EARLY_STOP_PATIENCE = 20

# Condot-aligned eval knobs (see condot_compat.evaluate.evaluate_biolord_condot_aligned):
SUBSAMPLE_CAP = 5000              # condot hardcoded cap; set 0 to disable
DIVIDE_BY_CONSTANT_NUM_CELL_TYPES = True   # True => divide per-pert metric by 3 (condot); False => by valid cell types
# biolord 的 unknown attribute 训练后 collapse 到 0, 如果直接用 mus 做预测, 每个 control cell 得到相同的向量,
# 分布指标 (MMD/FID/ED) 会严重不公. 打开 SAMPLE_FROM_GENERATIVE 后, 从 Normal(mus, sqrt(variances)) 采样,
# 复原 biolord 自己的生成分布, 让分布指标可比.
SAMPLE_FROM_GENERATIVE = True

In [3]:
adata = load_adata(PROCESSED_DATA_PATH)
print('Loaded:', adata.shape)

exp_dir = Path(RESULT_ROOT) / EXPERIMENT_TAG / f'seed{SEED}_bs{BATCH_SIZE}_ep{EPOCHS}_latent{N_LATENT}_ckpt{CHECKPOINT_FREQ}'
exp_dir.mkdir(parents=True, exist_ok=True)
print('Experiment dir:', exp_dir)

Loaded: (581777, 977)
Experiment dir: ../../../results/biolord_condot_aligned/biolord_sciplex3_condot_aligned/seed42_bs512_ep200_latent256_ckpt10


In [4]:
run_cfg = {
    'processed_data_path': PROCESSED_DATA_PATH,
    'result_root': RESULT_ROOT,
    'experiment_tag': EXPERIMENT_TAG,
    'gpu_id': GPU_ID,
    'seed': SEED,
    'split_key': SPLIT_KEY,
    'n_latent': N_LATENT,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'checkpoint_freq': CHECKPOINT_FREQ,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'checkpoint_dir': str(exp_dir / 'checkpoints'),
    'checkpoint_monitor': 'val_reconstruction_loss',
    'resume': True,
    'skip_if_complete': True,
    'load_best_on_end': True,
}
save_config_snapshot(run_cfg, str(exp_dir / 'config.json'))

In [5]:
model = train_biolord_model(
    adata=adata,
    split_key=SPLIT_KEY,
    model_name='biolord_sciplex3_condot_aligned',
    n_latent=N_LATENT,
    max_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    checkpoint_freq=CHECKPOINT_FREQ,
    num_workers=NUM_WORKERS,
    early_stopping_patience=EARLY_STOP_PATIENCE,
    seed=SEED,
    gpu_id=GPU_ID,
    module_params=module_params,
    trainer_params=trainer_params,
    checkpoint_dir=str(exp_dir / 'checkpoints'),
    checkpoint_monitor='val_reconstruction_loss',
    resume=True,
    skip_if_complete=True,
    load_best_on_end=True,
)

[biolord] Found completed run, loading: /home/jamin/Comparision/biolord_reproducibility/results/biolord_condot_aligned/biolord_sciplex3_condot_aligned/seed42_bs512_ep200_latent256_ckpt10/checkpoints/epoch=79-step=336640-val_reconstruction_loss=440.5124816894531
INFO     File                                                                                                      
         /home/jamin/Comparision/biolord_reproducibility/results/biolord_condot_aligned/biolord_sciplex3_condot_ali
         gned/seed42_bs512_ep200_latent256_ckpt10/checkpoints/epoch=79-step=336640-val_reconstruction_loss=440.5124
         816894531/model.pt already downloaded                                                                     


Seed set to 42


In [6]:
metrics = evaluate_biolord_condot_aligned(
    model=model,
    adata=adata,
    split_key=SPLIT_KEY,
    outdir=str(exp_dir),
    subsample_cap=SUBSAMPLE_CAP,
    divide_by_constant_num_cell_types=DIVIDE_BY_CONSTANT_NUM_CELL_TYPES,
    sample_from_generative=SAMPLE_FROM_GENERATIVE,
    seed=SEED,
)

metrics['summary']

INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        
INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


evaluating test: 100%|████████████████████████████████████████████| 178/178 [00:29<00:00,  6.07it/s]

INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        



evaluating ood: 100%|█████████████████████████████████████████████████| 9/9 [00:02<00:00,  3.77it/s]


,mmd,l2_loss,fid,r2,energy_distance
test_mean,0.040056,0.469668,1.989077,0.770855,0.026944
test_std,0.012245,0.138028,0.982087,0.157149,0.040547
ood_mean,0.020260,0.748618,1.945376,0.641243,0.250871
ood_std,0.015775,0.455352,1.772731,0.484469,0.314572


In [7]:
generate_umap_from_eval_data(str(exp_dir / 'eval'))
print('UMAP figures saved to:', exp_dir / 'eval' / 'plot')

UMAP figures saved to: ../../../results/biolord_condot_aligned/biolord_sciplex3_condot_aligned/seed42_bs512_ep200_latent256_ckpt10/eval/plot
